'''
Source: Impact4cast (Max Planck Institute)
Original code from the Max Planck Institute for Informatics / Max Planck Institute for Security and Privacy research group.

Modifications: Bug fixes for checkpoint resume mechanism and dataset adaptation for scientific entity impact prediction.
'''


In [ ]:
  # Comment
import os
import re
import time
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn 
import gc

# ---------------- Config ----------------
BASE_DIR = r"."
RESULT_FILE = os.path.join(BASE_DIR, "2019_train", "t0_c2_result", "All_AUC_Report_Year_Train_2019.txt")
MODEL_DIR = os.path.join(BASE_DIR, "2019_train", "t0_c2_net")
OUTPUT_DIR = os.path.join(BASE_DIR, "predictions")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Real prediction feature file (141 column normalized features)
FEATURE_FILE = os.path.join(BASE_DIR, "data_eval", "eval_data_pair_feature.parquet")

# Solution file containing original IDs and possible labels (contains v1, v2)
ID_FILE = os.path.join(BASE_DIR, "data_eval", "eval_data_pair_solution.parquet") 

BATCH_SIZE = 2048


# ---------------- Model definition ----------------
class ff_network(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ff_network, self).__init__()

        self.semnet = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        res = self.semnet(x)
        return res

# ---------------- Helper functions ----------------
def find_best_ir(result_file_path):
    pattern = re.compile(r'IR=\[(\d+)\]: train=([\d.]+); test=([\d.]+); eval=([\d.]+)')
    results = []
    with open(result_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                ir, train, test, eval_ = m.groups()
                results.append({
                    'IR': int(ir),
                    'train': float(train),
                    'test': float(test),
                    'eval': float(eval_)
                })
    if not results:
        raise RuntimeError(f"... {result_file_path} ... IR ...Check...")
    df = pd.DataFrame(results)
    best_ir = int(df.loc[df['eval'].idxmax(), 'IR'])
    best_eval = float(df['eval'].max())
    return best_ir, best_eval, df


def find_model_file_for_ir(model_dir, ir):
    ir_token = f"IR_{ir:03d}"
    candidates = [f for f in os.listdir(model_dir) if ir_token in f and f.endswith('.pt')]
    if not candidates:
        ir_token2 = f"IR_{ir}"
        candidates = [f for f in os.listdir(model_dir) if ir_token2 in f and f.endswith('.pt')]
    if not candidates:
        raise FileNotFoundError(f"... {model_dir} ...Found... {ir_token} ... .pt ...")
    candidates = sorted(candidates, key=lambda f: os.path.getmtime(os.path.join(model_dir, f)), reverse=True)
    return os.path.join(model_dir, candidates[0])

def inspect_model_state_dict(state_dict):
    """Check...state_dict...Input..."""
    print("\n📊 Check...state_dict...:")
    print(f"   state_dict...: {type(state_dict)}")
    if isinstance(state_dict, dict):
        print(f"   ...: {list(state_dict.keys())[:10]}")  # Code comment
        
        # Code comment
        possible_keys = [
            'semnet.0.weight',           # Code comment
            'module.semnet.0.weight',      # Comment
            '_orig_mod.semnet.0.weight', # Code comment
            'model.semnet.0.weight',     # Code comment
            'net.semnet.0.weight',       # Code comment
            'state_dict.semnet.0.weight' # Code comment
        ]
        
        for key in possible_keys:
            if key in state_dict:
                weight_shape = state_dict[key].shape
                if len(weight_shape) == 2:    # Comment
                    input_dim = weight_shape[1]
                    output_dim = weight_shape[0]
                    print(f"   ✅ ... '{key}' ...: Input...={input_dim}, Output...={output_dim}")
                    return input_dim
        
        # Code comment
        for key, value in state_dict.items():
            if 'weight' in key.lower() and value.ndim == 2:
                # Code comment
                input_dim = value.shape[1]
                print(f"   🔍 ... '{key}' (shape={value.shape}) ...Input...={input_dim}")
                return input_dim
    
    return None

  # Comment
class FuturePairsDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        
          # Comment
        self.id_cols = ['v1', 'v2'] 
        
        # Code comment
        non_features = set(self.id_cols)
        feature_cols = [c for c in df.columns if c not in non_features and np.issubdtype(df[c].dtype, np.number)]
            
        self.feature_cols = feature_cols
        
          # Comment
        if len(self.feature_cols) != 141:
            # Code comment
            print(f"⚠️ Warning: ...Extract... {len(self.feature_cols)}... 141...")
        else:
            print(f"✅ ...Extract...: {len(self.feature_cols)} ( columns '{self.feature_cols[0]}' ... '{self.feature_cols[-1]}')")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
          # Comment
        concept1_id = str(row[self.id_cols[0]])
        concept2_id = str(row[self.id_cols[1]])
            
          # Comment
        feat = row[self.feature_cols].values.astype(np.float32)
        pair = (concept1_id, concept2_id)
        
        return feat, pair


# ---------------- Main ----------------
def main():
      # Comment
    print("🔎 ... result ... ...")
    best_ir, best_eval, df_results = find_best_ir(RESULT_FILE)
    print(f"✅ Found... IR={best_ir} (eval={best_eval:.4f})")

      # Comment
    best_model_path = find_model_file_for_ir(MODEL_DIR, best_ir)
    print(f"✅ ...: {best_model_path}")

      # Comment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("🖥️ ...:", device)

      # Comment
    DEFAULT_HIDDEN_SIZE = 600
    DEFAULT_OUTPUT_SIZE = 1

      # Comment
    ck = torch.load(best_model_path, map_location='cpu')
    print(f"📦 Checkpoint...: {type(ck)}")
    
    # Code comment
    if isinstance(ck, dict):
        # Code comment
        candidate_keys = ['model_state_dict', 'state_dict', 'net', 'model', 'module']
        state_dict = None
        
        for key in candidate_keys:
            if key in ck:
                print(f"   Found '{key}' ...state_dict")
                state_dict = ck[key]
                break
        
        if state_dict is None:
            # Code comment
            # Code comment
            print("   ...Found...Filter......")
            state_dict = {k: v for k, v in ck.items() if isinstance(v, torch.Tensor)}
            if not state_dict:
                state_dict = ck  # Code comment
    else:
        state_dict = ck
    
      # Comment
    inferred_input_size = inspect_model_state_dict(state_dict)
    
    if inferred_input_size is None:
        # Code comment
        print("⚠️ ...Input [description] ...")
        if os.path.exists(FEATURE_FILE):
            df_test = pd.read_parquet(FEATURE_FILE)
            inferred_input_size = df_test.shape[1]
            print(f"✅ ...Input...: {inferred_input_size}")
        else:
            # Code comment
            print("⚠️ ...Input...: 141")
            inferred_input_size = 141
    
    input_size = inferred_input_size
    print(f"📐 ...Input... = {input_size}, ... = {DEFAULT_HIDDEN_SIZE}, Output = {DEFAULT_OUTPUT_SIZE}")

      # Comment
    model = ff_network(input_size, DEFAULT_HIDDEN_SIZE, DEFAULT_OUTPUT_SIZE)
    print("✅ ... ff_network ...")

      # Comment
    try:
        # Code comment
        model.load_state_dict(state_dict)
        print("✅ SuccessfullyLoadstate_dict")
    except Exception as e:
        print(f"⚠️ ...LoadFailed: {e}")
        try:
              # Comment
            new_state_dict = {}
            for k, v in state_dict.items():
                new_key = k.replace('module.', '')
                new_state_dict[new_key] = v
            model.load_state_dict(new_state_dict)
            print("✅ SuccessfullyLoadstate_dict (Remove 'module.' ...)")
        except Exception as e2:
            print(f"❌ LoadFailed: {e2}")
            raise RuntimeError(f"...Load...: {e2}")

    model.to(device)
    model.eval()
    print("✅ ...Load...Set...")

      # Comment
    print("\n📂 Load......")
    
      # Comment
    if not os.path.exists(FEATURE_FILE):
        raise FileNotFoundError(f"...Found...: {FEATURE_FILE}")
    df_features = pd.read_parquet(FEATURE_FILE)
    print(f"✅ ...Load: {df_features.shape}")
    
      # Comment
    if not os.path.exists(ID_FILE):
        raise FileNotFoundError(f"...Found ID ...: {ID_FILE}")
    df_ids = pd.read_parquet(ID_FILE)[['v1', 'v2']]  # Code comment
    print(f"✅ ID...Load: {df_ids.shape}")
    
      # Comment
    if len(df_features) != len(df_ids):
        raise ValueError(f"... ({len(df_features)}  rows) ... ID ... ({len(df_ids)}  rows)  rows [description] ")
        
      # Comment
    df_ids = df_ids.reset_index(drop=True)
    df_features = df_features.reset_index(drop=True)

    # Code comment
    df_future = pd.concat([df_ids, df_features], axis=1)
    
    del df_ids, df_features
    gc.collect()

    print(f"✅ ...: {df_future.shape[0]}  rows, {df_future.shape[1]}  columns")
    print(f"    columns...: {df_future.columns[:5].tolist()} ...")

      # Comment
    print("\n🔮 Start......")
    ds = FuturePairsDataset(df_future)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    scores, pairs = [], []

    start_t = time.time()
    with torch.no_grad():
        for i, (x_batch, pair_batch) in enumerate(loader):
            x = x_batch.to(device)

            # Code comment
            expected_input_dim = model.semnet[0].in_features
            current_input_dim = x.shape[1]
            
            if current_input_dim != expected_input_dim:
                print(f"⚠️ ... {i}: Input...! ... {expected_input_dim}, ... {current_input_dim}")
                if current_input_dim < expected_input_dim:
                    # Code comment
                    padding = torch.zeros(x.shape[0], expected_input_dim - current_input_dim).to(device)
                    x = torch.cat([x, padding], dim=1)
                    print(f"   ... {expected_input_dim} ...")
                else:
                    # Code comment
                    x = x[:, :expected_input_dim]
                    print(f"   ... {expected_input_dim} ...")
            
            out = model(x)
            if out.ndim == 2 and out.shape[1] == 1:
                probs = torch.sigmoid(out).squeeze(dim=1)
            elif out.ndim == 1:
                probs = torch.sigmoid(out)
            elif out.ndim == 2 and out.shape[1] == 2:
                probs = F.softmax(out, dim=1)[:, 1]
            else:
                probs = torch.sigmoid(out).squeeze()

            scores.extend(probs.cpu().numpy().tolist())
            
            c1_ids = pair_batch[0]
            c2_ids = pair_batch[1]
            pairs.extend(list(zip(c1_ids, c2_ids)))
            
            if (i+1) % 10 == 0:
                print(f"   Processed {i+1} ..., ... {time.time()-start_t:.1f}s")

      # Comment
    print("\n💾 Save...Result...")
    df_out = pd.DataFrame(pairs, columns=['concept1', 'concept2'])
    df_out['score'] = scores
    df_out = df_out.sort_values('score', ascending=False).reset_index(drop=True)
    out_csv = os.path.join(OUTPUT_DIR, "future_concept_pairs_sorted.csv")
    df_out.to_csv(out_csv, index=False, encoding='utf-8-sig')
    print(f"✅ Result...Save: {out_csv}")
    print(f"📊 ...Done! ... {len(df_out)} ...")
    print("\n🏆 Top 10 ...Result:")
    print(df_out.head(10))
    
      # Comment
    print(f"\n📈 ...Statistics:")
    print(f"   ...: {df_out['score'].min():.4f}")
    print(f"   ...: {df_out['score'].max():.4f}")
    print(f"   ...: {df_out['score'].mean():.4f}")
    print(f"   ...: {df_out['score'].median():.4f}")


if __name__ == "__main__":
    main()

In [ ]:
  # Comment
import os
import re
import time
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn 
import gc
import math

# ---------------- Config ----------------
BASE_DIR = r"."
RESULT_FILE = os.path.join(BASE_DIR, "2019_train", "t0_c2_result", "All_AUC_Report_Year_Train_2019.txt")
MODEL_DIR = os.path.join(BASE_DIR, "2019_train", "t0_c2_net")
OUTPUT_DIR = os.path.join(BASE_DIR, "predictions")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Real prediction feature file (141 column normalized features)
FEATURE_FILE = os.path.join(BASE_DIR, "data_eval", "eval_data_pair_feature.parquet")

# Solution file containing original IDs and possible labels (contains v1, v2)
ID_FILE = os.path.join(BASE_DIR, "data_eval", "data_eval_pair_solution.parquet") 

BATCH_SIZE = 2048


  # Comment
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

class FeatureProjection(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(FeatureProjection, self).__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.projection(x)

class FeatureAggregator(nn.Module):
    def __init__(self, d_model, nhead=8):
        super(FeatureAggregator, self).__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, query, key_value):
        attn_output, _ = self.cross_attn(query, key_value, key_value)
        return self.norm(query + attn_output)

class OutputLayer(nn.Module):
    def __init__(self, d_model, output_dim=1):
        super(OutputLayer, self).__init__()
        self.output_net = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model // 4),
            nn.ReLU(),
            nn.Linear(d_model // 4, output_dim)
        )
    
    def forward(self, x):
        return self.output_net(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim=141, d_model=256, nhead=8, num_layers=4, output_dim=1):
        super(TransformerModel, self).__init__()
        
        # Code comment
        self.feature_proj = FeatureProjection(input_dim, d_model)
        
        # Code comment
        self.pos_encoder = PositionalEncoding(d_model)
        
          # Comment
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Code comment
        self.feature_aggregator = FeatureAggregator(d_model, nhead)
        
          # Comment
        self.output_layer = OutputLayer(d_model, output_dim)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        # Code comment
        if x.dim() == 2:
            x = x.unsqueeze(1)  # (batch_size, 1, input_dim)
        
        # Code comment
        x = self.feature_proj(x)  # (batch_size, seq_len, d_model)
        
        # Code comment
        x = self.pos_encoder(x)
        
          # Comment
        x = self.transformer_encoder(x)
        
        # Code comment
        # Code comment
        aggregated = self.feature_aggregator(x[:, 0:1, :], x)
        
        # Output
        output = self.output_layer(aggregated.squeeze(1))
        
        return output

# ---------------- Helper functions ----------------
def find_best_ir(result_file_path):
    pattern = re.compile(r'IR=\[(\d+)\]: train=([\d.]+); test=([\d.]+); eval=([\d.]+)')
    results = []
    with open(result_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                ir, train, test, eval_ = m.groups()
                results.append({
                    'IR': int(ir),
                    'train': float(train),
                    'test': float(test),
                    'eval': float(eval_)
                })
    if not results:
        raise RuntimeError(f"... {result_file_path} ... IR ...")
    df = pd.DataFrame(results)
    best_ir = int(df.loc[df['eval'].idxmax(), 'IR'])
    best_eval = float(df['eval'].max())
    return best_ir, best_eval, df

def find_model_file_for_ir(model_dir, ir):
    ir_token = f"IR_{ir:03d}"
    candidates = [f for f in os.listdir(model_dir) if ir_token in f and f.endswith('.pt')]
    if not candidates:
        ir_token2 = f"IR_{ir}"
        candidates = [f for f in os.listdir(model_dir) if ir_token2 in f and f.endswith('.pt')]
    if not candidates:
        raise FileNotFoundError(f"... {model_dir} ...Found... {ir_token} ... .pt ...")
    candidates = sorted(candidates, key=lambda f: os.path.getmtime(os.path.join(model_dir, f)), reverse=True)
    return os.path.join(model_dir, candidates[0])

# ---------------- Dataset ----------------
class FuturePairsDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.id_cols = ['v1', 'v2']
        
        # Code comment
        non_features = set(self.id_cols)
        feature_cols = [c for c in df.columns if c not in non_features and np.issubdtype(df[c].dtype, np.number)]
        self.feature_cols = feature_cols
        
        if len(self.feature_cols) != 141:
            print(f"⚠️ Warning: ... {len(self.feature_cols)}... 141")
        else:
            print(f"✅ ...: {len(self.feature_cols)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        concept1_id = str(row[self.id_cols[0]])
        concept2_id = str(row[self.id_cols[1]])
        feat = row[self.feature_cols].values.astype(np.float32)
        pair = (concept1_id, concept2_id)
        return feat, pair

# ---------------- Main ----------------
def main():
      # Comment
    print("🔎 ... result ... ...")
    best_ir, best_eval, df_results = find_best_ir(RESULT_FILE)
    print(f"✅ Found... IR={best_ir} (eval={best_eval:.4f})")

      # Comment
    best_model_path = find_model_file_for_ir(MODEL_DIR, best_ir)
    print(f"✅ ...: {best_model_path}")

      # Comment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🖥️ ...: {device}")

    # 4) Loadcheckpoint
    print("📦 Load......")
    checkpoint = torch.load(best_model_path, map_location='cpu')
    
      # Comment
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
        print("   ... checkpoint ...Extract model_state_dict")
    elif isinstance(checkpoint, dict) and any('transformer_encoder' in k for k in checkpoint.keys()):
        state_dict = checkpoint
        print("   ... checkpoint ... state_dict")
    elif isinstance(checkpoint, nn.Module):
        state_dict = checkpoint.state_dict()
        print("   checkpoint ... nn.Module...Extract state_dict")
    else:
        state_dict = checkpoint
        print(f"   checkpoint ...: {type(checkpoint)}")
    
      # Comment
    print("🏗️ Build Transformer ......")
    # Code comment
    d_model = None
    if 'feature_proj.projection.0.weight' in state_dict:
        d_model = state_dict['feature_proj.projection.0.weight'].shape[0]
    
    num_layers = 0
    for key in state_dict.keys():
        if 'transformer_encoder.layers.' in key:
            layer_num = int(key.split('.')[2])
            num_layers = max(num_layers, layer_num + 1)
    
    print(f"   ...Parameters: d_model={d_model}, num_layers={num_layers}")
    
      # Comment
    model = TransformerModel(
        input_dim=141,
        d_model=d_model or 256,
        nhead=8,
        num_layers=num_layers or 4,
        output_dim=1
    )
    
      # Comment
    try:
          # Comment
        new_state_dict = {}
        for k, v in state_dict.items():
            new_key = k.replace('module.', '')
            new_state_dict[new_key] = v
        
          # Comment
        missing_keys, unexpected_keys = model.load_state_dict(new_state_dict, strict=False)
        
        if missing_keys:
            print(f"⚠️ ...: {missing_keys[:5]}...")
        if unexpected_keys:
            print(f"⚠️ ...: {unexpected_keys[:5]}...")
        
        print("✅ ...LoadSuccessfully")
    except Exception as e:
        print(f"❌ LoadFailed: {e}")
        raise
    
    model.to(device)
    model.eval()
    
      # Comment
    print("\n📂 Load......")
    
    if not os.path.exists(FEATURE_FILE):
        raise FileNotFoundError(f"...Found...: {FEATURE_FILE}")
    df_features = pd.read_parquet(FEATURE_FILE)
    print(f"✅ ...: {df_features.shape}")
    
    if not os.path.exists(ID_FILE):
        raise FileNotFoundError(f"...Found ID ...: {ID_FILE}")
    df_ids = pd.read_parquet(ID_FILE)[['v1', 'v2']]
    print(f"✅ ID...: {df_ids.shape}")
    
    if len(df_features) != len(df_ids):
        raise ValueError(f" rows...: ... {len(df_features)} vs ID {len(df_ids)}")
    
    df_ids = df_ids.reset_index(drop=True)
    df_features = df_features.reset_index(drop=True)
    df_future = pd.concat([df_ids, df_features], axis=1)
    
    del df_ids, df_features
    gc.collect()
    
    print(f"✅ ...: {len(df_future)}  rows")
    
      # Comment
    print("\n🔮 Start......")
    ds = FuturePairsDataset(df_future)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    scores, pairs = [], []
    
    start_t = time.time()
    with torch.no_grad():
        for i, (x_batch, pair_batch) in enumerate(loader):
            x = x_batch.to(device)
            
            # Code comment
            out = model(x)
            
            # Code comment
            if out.ndim == 2 and out.shape[1] == 1:
                probs = torch.sigmoid(out).squeeze(dim=1)
            elif out.ndim == 1:
                probs = torch.sigmoid(out)
            elif out.ndim == 2 and out.shape[1] == 2:
                probs = F.softmax(out, dim=1)[:, 1]
            else:
                probs = torch.sigmoid(out).squeeze()
            
            scores.extend(probs.cpu().numpy().tolist())
            
            c1_ids = pair_batch[0]
            c2_ids = pair_batch[1]
            pairs.extend(list(zip(c1_ids, c2_ids)))
            
            if (i+1) % 10 == 0:
                print(f"   Processed {i+1} ... ({len(scores)} ...), ... {time.time()-start_t:.1f}s")
    
    # 9) SaveResult
    print("\n💾 Save...Result...")
    df_out = pd.DataFrame(pairs, columns=['concept1', 'concept2'])
    df_out['score'] = scores
    df_out = df_out.sort_values('score', ascending=False).reset_index(drop=True)
    out_csv = os.path.join(OUTPUT_DIR, "future_concept_pairs_sorted.csv")
    df_out.to_csv(out_csv, index=False, encoding='utf-8-sig')
    
    print(f"✅ Result...Save: {out_csv}")
    print(f"📊 ... {len(df_out)} ...")
    print("\n🏆 Top 20 ...Result:")
    print(df_out.head(20))
    
    print(f"\n📈 ...Statistics:")
    print(f"   ...: {df_out['score'].min():.6f}")
    print(f"   ...: {df_out['score'].max():.6f}")
    print(f"   ...: {df_out['score'].mean():.6f}")
    print(f"   ...: {df_out['score'].median():.6f}")
    
      # Comment
    print(f"\n📊 ...:")
    print(f"   >0.9: {(df_out['score'] > 0.9).sum()}")
    print(f"   0.8-0.9: {((df_out['score'] > 0.8) & (df_out['score'] <= 0.9)).sum()}")
    print(f"   0.7-0.8: {((df_out['score'] > 0.7) & (df_out['score'] <= 0.8)).sum()}")
    print(f"   0.6-0.7: {((df_out['score'] > 0.6) & (df_out['score'] <= 0.7)).sum()}")
    print(f"   <=0.6: {(df_out['score'] <= 0.6).sum()}")


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import os

# ---------------- Configuration ----------------
BASE_DIR = r"."
# Code comment
INPUT_CSV_PATH = os.path.join(BASE_DIR, "predictions", "future_concept_pairs_sorted.csv")
# Code comment
CONCEPT_FILE_PATH = r"./data/entities.txt"
  # Comment
OUTPUT_CSV_PATH = os.path.join(BASE_DIR, "predictions", "top10_with_keywords.csv")

# Ensure the file exists before attempting to read it
if os.path.exists(INPUT_CSV_PATH):
    print(f"Reading the first 10 lines of the file: {INPUT_CSV_PATH}")
    with open(INPUT_CSV_PATH, 'r', encoding='utf-8') as file:
        for i in range(10):
            print(file.readline().strip())
else:
    print(f"❌ File not found: {INPUT_CSV_PATH}")

In [ ]:
import pandas as pd
import os

# ---------------- Configuration ----------------
BASE_DIR = r"."
# Code comment
INPUT_CSV_PATH = os.path.join(BASE_DIR, "predictions", "future_concept_pairs_sorted.csv")
# Code comment
CONCEPT_FILE_PATH =r"./data/entities.txt"
  # Comment
OUTPUT_CSV_PATH = os.path.join(BASE_DIR, "predictions", "top10_with_keywords.csv")

# Code comment
top_10_data = {
    'concept1': [4777.0, 19776.0, 15794.0, 11133.0, 12311.0, 17840.0, 18947.0, 226.0, 6431.0, 27432.0],
    'concept2': [5597.0, 29060.0, 35278.0, 35171.0, 24526.0, 35309.0, 23462.0, 5383.0, 23701.0, 29417.0],
    'score': [0.750425, 0.747279, 0.746214, 0.746155, 0.745913, 0.745905, 0.745553, 0.745519, 0.744880, 0.744153]
}

  # Comment

def load_concept_map(file_path):
    """
    Load full_domain_concepts.txt ...Create ID (...) ...
    ... rows... items... 0 Start...
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ ...does not exist...{file_path}")

    print(f"✅ Loading...: {file_path}")
    concept_map = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        # Code comment
        for i, line in enumerate(f):
            keyword = line.strip()
            # Code comment
            concept_map[i] = keyword
            
    print(f"✅ ...Load {len(concept_map)}  items...")
    return concept_map

def process_top_results(input_df, concept_map, output_path):
    """
    Process Top 10 Result...Save to CSV...
    """
    # Code comment
    df = input_df.copy()
    df['concept1_id'] = df['concept1'].astype(int)
    df['concept2_id'] = df['concept2'].astype(int)
    
    # Code comment
    # Code comment
    df['concept1_keyword'] = df['concept1_id'].map(concept_map).fillna('Keyword Not Found')
    df['concept2_keyword'] = df['concept2_id'].map(concept_map).fillna('Keyword Not Found')

    # Code comment
    final_df = df[['concept1_id', 'concept1_keyword', 'concept2_id', 'concept2_keyword', 'score']]
    
    # SaveResult
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ SuccessfullyGenerate... CSV ...{output_path}")
    print("\n--- Top 10 ...Result... ---")
    print(final_df.head(10).to_markdown(index=False))
    
    return final_df

  # Comment
try:
      # Comment
    concept_map = load_concept_map(CONCEPT_FILE_PATH)
    
      # Comment
    if os.path.exists(INPUT_CSV_PATH):
        print(f"...Read...Result...: {INPUT_CSV_PATH}")
        df_results = pd.read_csv(INPUT_CSV_PATH).head(10) # Code comment
    else:
        # Code comment
        print(f"⚠️ ...Result CSV ...does not exist ({INPUT_CSV_PATH})... rows...")
        df_results = pd.DataFrame(top_10_data)

      # Comment
    process_top_results(df_results, concept_map, OUTPUT_CSV_PATH)

except FileNotFoundError as e:
    print(f"\n❌ ...Error...{e}")
except Exception as e:
    print(f"\n❌ ...Error...{e}")

  # Comment
print("\n--- ...OpenGenerate... CSV ...View...Result ---")

In [ ]:
import pandas as pd
import os

# ---------------- Configuration ----------------
BASE_DIR = r"."

  # Comment
INPUT_CSV_PATH = os.path.join(BASE_DIR, "predictions", "future_concept_pairs_sorted.csv")

  # Comment
CONCEPT_FILE_PATH = r"./data/entities.txt"

  # Comment
OUTPUT_CSV_PATH = os.path.join(BASE_DIR, "predictions", "all_predictions_with_entities.csv")

  # Comment

def load_concept_map(file_path):
    """
    Load full_domain_concepts.txt ...Create ID (...) ...
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ ...does not exist...{file_path}")

    print(f"⏳ Loading...: {file_path}")
    concept_map = {}
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            # Code comment
            for i, line in enumerate(f):
                keyword = line.strip()
                if keyword: # Code comment
                    concept_map[i] = keyword
        print(f"✅ ...Load {len(concept_map)}  items...")
        return concept_map
    except Exception as e:
        raise Exception(f"Read...Failed: {e}")

def process_all_results(input_path, concept_map, output_path):
    """
    Read [description] Save...
    """
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"❌ ...Result...{input_path}")

    print(f"⏳ ...Read...Result CSV (...)...")
      # Comment
    df = pd.read_csv(input_path)
    print(f"✅ ReadSuccessfully... {len(df)} ...")

    print("⏳ ... rows......")
      # Comment
    # Code comment
    df['concept1_id'] = df['concept1'].fillna(-1).astype(int)
    df['concept2_id'] = df['concept2'].fillna(-1).astype(int)
    
      # Comment
    # Code comment
    df['concept1_keyword'] = df['concept1_id'].map(concept_map).fillna('Unknown ID')
    df['concept2_keyword'] = df['concept2_id'].map(concept_map).fillna('Unknown ID')

      # Comment
    final_df = df[['concept1_id', 'concept1_keyword', 'concept2_id', 'concept2_keyword', 'score']]
    
    print(f"⏳ ...SaveResult...: {output_path}")
    # SaveResult
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"✅ SaveDone...")
    
    return final_df

  # Comment
if __name__ == "__main__":
    try:
          # Comment
        concept_map = load_concept_map(CONCEPT_FILE_PATH)
        
          # Comment
        process_all_results(INPUT_CSV_PATH, concept_map, OUTPUT_CSV_PATH)

        print("\n" + "="*50)
        print(f"Process [description] Result...Save...\n{OUTPUT_CSV_PATH}")
        print("="*50)

    except FileNotFoundError as e:
        print(f"\n❌ ...Error...{e}")
        print("...Check BASE_DIR ...")
    except Exception as e:
        print(f"\n❌ ... rows...{e}")

In [ ]:
# generate_train_data_2022_2025.py
import os
import re
import time
import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pickle
import gzip
import gc
from datetime import datetime, date
import warnings
import torch.nn as nn  
warnings.filterwarnings('ignore')

# ---------------- Config ----------------
BASE_DIR = r"."
RESULT_FILE = os.path.join(BASE_DIR, "2019_train", "t0_c2_result", "All_AUC_Report_Year_Train_2019.txt")
MODEL_DIR = os.path.join(BASE_DIR, "2019_train", "t0_c2_net")
OUTPUT_DIR = os.path.join(BASE_DIR, "data_for_train_2025")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Code comment
FEATURE_FOLDER = os.path.join(BASE_DIR, "data_for_features")
DYNAMIC_GRAPH_FILE = os.path.join(BASE_DIR, "data_concept_graph", "full_dynamic_graph.parquet")
CONCEPT_LIST_FILE = r"./data/entities.txt" # Code comment

# Code comment
FEATURE_FILE = os.path.join(BASE_DIR, "data_eval", "eval_data_pair_feature.parquet")
ID_FILE = os.path.join(BASE_DIR, "data_eval", "data_eval_pair_solution.parquet")

# Code comment
YEAR_START = 2022  # Code comment
YEAR_END = 2025    # Code comment
YEARS_DELTA = 3    # Code comment

# Code comment
BATCH_SIZE = 1000000  # Code comment
MAX_PAIRS = None    # Code comment

  # Comment
class ff_network(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ff_network, self).__init__()
        self.semnet = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        return self.semnet(x)


  # Comment
def find_best_ir(result_file_path):
    """...Result...Found...IR...eval AUC..."""
    pattern = re.compile(r'IR=\[(\d+)\]: train=([\d.]+); test=([\d.]+); eval=([\d.]+)')
    results = []
    
    print(f"🔎 ...Result...: {result_file_path}")
    with open(result_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                ir, train, test, eval_ = m.groups()
                results.append({
                    'IR': int(ir),
                    'train': float(train),
                    'test': float(test),
                    'eval': float(eval_)
                })
    
    if not results:
        raise RuntimeError(f"... {result_file_path} ...IR...Check...")
    
    df = pd.DataFrame(results)
    best_idx = df['eval'].idxmax()
    best_ir = int(df.loc[best_idx, 'IR'])
    best_eval = float(df.loc[best_idx, 'eval'])
    
    print(f"✅ Found...IR={best_ir} (eval AUC={best_eval:.4f})")
    print("\n...IR...AUC...:")
    print(df.to_string(index=False))
    
    return best_ir, best_eval, df


def load_feature_mapping(feature_file, id_file):
    """
    Load...ID...Create...
    
    ...:
        feature_map: ...(v1, v2)...141...
    """
    print(f"🔎 Load......")
    
      # Comment
    if not os.path.exists(feature_file):
        raise FileNotFoundError(f"...does not exist: {feature_file}")
    
    print(f"Load...: {feature_file}")
    df_features = pd.read_parquet(feature_file)
    print(f"...: {df_features.shape}")
    
      # Comment
    if not os.path.exists(id_file):
        raise FileNotFoundError(f"ID...does not exist: {id_file}")
    
    print(f"LoadID...: {id_file}")
    df_ids = pd.read_parquet(id_file)
    
    # Code comment
    if 'v1' in df_ids.columns and 'v2' in df_ids.columns:
        df_ids = df_ids[['v1', 'v2']]
    else:
        # Code comment
        df_ids = df_ids.iloc[:, :2]
        df_ids.columns = ['v1', 'v2']
    
    print(f"ID...: {df_ids.shape}")
    
      # Comment
    if len(df_features) != len(df_ids):
        raise ValueError(f"...({len(df_features)} rows)...ID...({len(df_ids)} rows) rows...")
    
      # Comment
    df_combined = pd.concat([df_ids.reset_index(drop=True), 
                             df_features.reset_index(drop=True)], axis=1)
    
      # Comment
    feature_map = {}
    
    print("Create......")
    for idx, row in df_combined.iterrows():
        v1, v2 = int(row['v1']), int(row['v2'])
        # Code comment
        if v1 > v2:
            v1, v2 = v2, v1
        
          # Comment
        features = row.iloc[2:].values.astype(np.float32)
        
        # Code comment
        if len(features) != 141:
            print(f"⚠️ Warning: ... {len(features)}...141")
        
        feature_map[(v1, v2)] = features
    
    print(f"✅ SuccessfullyCreate {len(feature_map)}  items...")
    
      # Comment
    if len(feature_map) > 0:
        sample_features = next(iter(feature_map.values()))
        print(f"...: {len(sample_features)}")
        print(f"...: [{sample_features.min():.4f}, {sample_features.max():.4f}]")
        print(f"...: {sample_features.mean():.4f}")
    
    return feature_map


def get_unconnected_pairs_batch(graph_file, year_start, batch_size=100000, max_pairs=None):
    """
    ...year_start...
    
    ...: Generate... (batch_pairs, total_count)
    """
    print(f"🔎 ... {year_start}......")
    
      # Comment
    pf = pq.ParquetFile(graph_file)
    
    # Code comment
    existing_pairs = set()
    total_edges = 0
    
    print("...already exists......")
    for batch in pf.iter_batches(batch_size=batch_size):
        df_batch = batch.to_pandas()
        
          # Comment
        time_col = None
        for col in df_batch.columns:
            if 'time' in col.lower() or 'date' in col.lower() or 'year' in col.lower():
                time_col = col
                break
        
        # Code comment
        if time_col is not None:
            try:
                # Code comment
                if pd.api.types.is_integer_dtype(df_batch[time_col]):
                    df_batch = df_batch[df_batch[time_col] <= year_start]
                # Code comment
                elif pd.api.types.is_string_dtype(df_batch[time_col]):
                    year_start_str = str(year_start)
                    df_batch = df_batch[df_batch[time_col].str.startswith(year_start_str) | 
                                         (df_batch[time_col] < str(year_start + 1))]
                # Code comment
                else:
                    year_start_date = pd.Timestamp(f"{year_start}-12-31")
                    df_batch = df_batch[pd.to_datetime(df_batch[time_col]) <= year_start_date]
            except Exception as e:
                print(f"  ...: {e}...Skip...")
        
          # Comment
        for _, row in df_batch.iterrows():
            v1, v2 = row['v1'], row['v2']
            # Code comment
            if v1 < v2:
                existing_pairs.add((v1, v2))
            else:
                existing_pairs.add((v2, v1))
        
        total_edges += len(df_batch)
        if len(existing_pairs) % 100000 == 0:
            print(f"  ... {len(existing_pairs)}  items...")
    
    print(f"✅ ... {len(existing_pairs)}  itemsalready exists...{year_start}...")
    
      # Comment
    # Code comment
    concepts = set()
    for year in [year_start, year_start-1, year_start-2]:
        cite_file = os.path.join(FEATURE_FOLDER, f"concept_node_citation_data_{year}.parquet")
        if os.path.exists(cite_file):
            try:
                df_cite = pd.read_parquet(cite_file)
                # Code comment
                concepts.update(df_cite.iloc[:, 0].tolist())
            except Exception as e:
                print(f"  Load {year}...Failed: {e}")
    
    concepts = list(concepts)
    if len(concepts) == 0:
        # Code comment
        for v1, v2 in existing_pairs:
            concepts.add(v1)
            concepts.add(v2)
        concepts = list(concepts)
    
    print(f"✅ ... {len(concepts)}  items...")
    
      # Comment
    from itertools import combinations
    
    total_possible = len(concepts) * (len(concepts) - 1) // 2
    print(f"📊  [description] : {total_possible:,}")
    
    # Code comment
    #max_to_process = max_pairs if max_pairs else min(total_possible, 1000000)  # Code comment
      # Comment
    
    # Code comment
    unconnected_batch = []
    batch_count = 0
    total_unconnected = 0
    
    for i, (v1, v2) in enumerate(combinations(concepts, 2)):
        # Code comment
        if v1 > v2:
            v1, v2 = v2, v1
        
          # Comment
        if (v1, v2) not in existing_pairs:
            unconnected_batch.append((v1, v2))
            total_unconnected += 1
        
        # Code comment
        if len(unconnected_batch) >= batch_size or (max_pairs and total_unconnected >= max_pairs):
            if unconnected_batch:  # Code comment
                batch_count += 1
                print(f"  Generate... {batch_count}: {len(unconnected_batch)}  items...")
                yield unconnected_batch, total_unconnected
                unconnected_batch = []
        
        # Code comment
        if max_pairs and total_unconnected >= max_pairs:
            break
        
        # Code comment
        if (i + 1) % 100000 == 0:
            print(f"  Processed {i+1:,}  items...Found {total_unconnected:,}  items...")
    
    # Code comment
    if unconnected_batch:
        batch_count += 1
        print(f"  Generate... {batch_count}: {len(unconnected_batch)}  items...")
        yield unconnected_batch, total_unconnected
    
    print(f"✅ ...Found {total_unconnected:,}  items...")


def get_label_for_pair(v1, v2, graph_file, year_end, IR):
    """
    ...year_end...
    
    ...year_end... ≥ IR...=1...=0
    """
    try:
        pf = pq.ParquetFile(graph_file)
    except Exception as e:
        print(f"  ...Open...: {e}")
        return 0, 0
    
    total_citations = 0
    
    # Code comment
    for batch in pf.iter_batches(batch_size=100000):
        try:
            df_batch = batch.to_pandas()
        except:
            continue
        
        # Code comment
        mask = ((df_batch['v1'] == v1) & (df_batch['v2'] == v2)) | \
               ((df_batch['v1'] == v2) & (df_batch['v2'] == v1))
        
        if mask.any():
            df_pair = df_batch[mask]
            
            # Code comment
            cite_col = None
            for col in df_pair.columns:
                if col.lower().startswith('c') and col[1:].isdigit():
                    year = int(col[1:])
                    if year <= year_end:
                        cite_col = col
                        break
                elif col.lower() in ['ct', 'citation', 'citations']:
                    cite_col = col
            
            if cite_col is not None:
                total_citations += df_pair[cite_col].sum()
    
    label = 1 if total_citations >= IR else 0
    return label, total_citations


def extract_features_from_mapping(v1, v2, feature_map):
    """
    ...Extract...(v1, v2)...141...
    
    Parameters:
        v1, v2: ...ID
        feature_map: ...(v1, v2)...141...
    
    ...:
        141... columns [description] None
    """
    # Code comment
    if v1 > v2:
        v1, v2 = v2, v1
    
    # Code comment
    if (v1, v2) in feature_map:
        features = feature_map[(v1, v2)]
        return features.tolist() if isinstance(features, np.ndarray) else features
    else:
        return None


# ---------------- Main ----------------
def main():
    start_time_total = time.time()
    
      # Comment
    print("=" * 60)
    print("...1: ...IR...")
    print("=" * 60)
    best_ir, best_eval, df_results = find_best_ir(RESULT_FILE)
    
      # Comment
    print("\n" + "=" * 60)
    print("...2: Load...")
    print("=" * 60)
    feature_map = load_feature_mapping(FEATURE_FILE, ID_FILE)
    
      # Comment
    output_file = os.path.join(OUTPUT_DIR, f"train_data_{YEAR_START}_{YEAR_END}_IR{best_ir}.parquet")
    temp_dir = os.path.join(OUTPUT_DIR, "temp")
    os.makedirs(temp_dir, exist_ok=True)
    
    print("\n" + "=" * 60)
    print(f"...3: Generate...")
    print(f"Output...: {output_file}")
    print(f"...: {YEAR_START}")
    print(f"...: {YEAR_END}")
    print(f"IR...: {best_ir}")
    print("=" * 60)
    
      # Comment
    print("\n📊 Start...Process......")
    
    batch_files = []
    total_processed = 0
    batch_num = 0
    total_found_in_map = 0
    total_not_found = 0
    
    # Code comment
    pair_generator = get_unconnected_pairs_batch(
        DYNAMIC_GRAPH_FILE, 
        YEAR_START, 
        batch_size=BATCH_SIZE,
        max_pairs=MAX_PAIRS
    )
    
    for batch_pairs, total_found in pair_generator:
        batch_num += 1
        batch_start = time.time()
        
        print(f"\n--- Process... {batch_num} ---")
        print(f"...: {len(batch_pairs)}")
        print(f"CumulativeFound...: {total_found}")
        
          # Comment
        batch_data = []
        batch_found = 0
        batch_not_found = 0
        
        for i, (v1, v2) in enumerate(batch_pairs):
            if (i + 1) % 100 == 0:  # Code comment
                print(f"  Process... {i+1}/{len(batch_pairs)}  items...")
            
            # Code comment
            features = extract_features_from_mapping(v1, v2, feature_map)
            
            if features is not None:
                # Code comment
                label, citations = get_label_for_pair(v1, v2, DYNAMIC_GRAPH_FILE, YEAR_END, best_ir)
                
                  # Comment
                row = [v1, v2, label] + features
                batch_data.append(row)
                
                batch_found += 1
                total_found_in_map += 1
            else:
                batch_not_found += 1
                total_not_found += 1
            
            total_processed += 1
        
          # Comment
        if batch_data:
            df_batch = pd.DataFrame(batch_data)
              # Comment
            col_names = ['v1', 'v2', 'label'] + [f'f{i}' for i in range(141)]
            df_batch.columns = col_names
            
            temp_file = os.path.join(temp_dir, f"batch_{batch_num:04d}.parquet")
            df_batch.to_parquet(temp_file, index=False)
            batch_files.append(temp_file)
            
            batch_time = time.time() - batch_start
            print(f"✅ ... {batch_num} Done... {batch_time:.2f}s...Save... {temp_file}")
            print(f"   ...Found: {batch_found}  items...Found: {batch_not_found}  items")
            print(f"   ...: {df_batch['label'].value_counts().to_dict()}")
            
            # Code comment
            del df_batch, batch_data
            gc.collect()
        else:
            print(f"⚠️ ... {batch_num} ...Found...")
    
    print(f"\n📊 Statistics...:")
    print(f"   ...Process...: {total_processed}")
    print(f"   ...Found: {total_found_in_map}")
    print(f"   ...Found: {total_not_found}")
    print(f"   Found...: {total_found_in_map/total_processed*100:.2f}%" if total_processed > 0 else "   Found...: N/A")
    
      # Comment
    if batch_files:
        print("\n" + "=" * 60)
        print("...4: Merge...")
        print("=" * 60)
        
          # Comment
        dfs = []
        for i, f in enumerate(batch_files):
            print(f"Read... {i+1}/{len(batch_files)}: {f}")
            df = pd.read_parquet(f)
            dfs.append(df)
            
            # Code comment
            # os.remove(f)
        
        print(f"Merge {len(dfs)}  itemsDataFrame...")
        df_final = pd.concat(dfs, ignore_index=True)
        
        # Code comment
        df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
        
        print(f"...Size: {len(df_final)}  rows, {len(df_final.columns)}  columns")
        print(f"...:")
        print(df_final['label'].value_counts())
        print(f"...: {df_final['label'].mean():.4f}")
        
          # Comment
        print(f"Save...: {output_file}")
        df_final.to_parquet(output_file, index=False)
        
          # Comment
        sample_file = output_file.replace('.parquet', '_sample.csv')
        df_final.head(1000).to_csv(sample_file, index=False)
        print(f"...Save...: {sample_file}")
        
        # CleanTemp file
        print("CleanTemp file...")
        for f in batch_files:
            try:
                os.remove(f)
                print(f"  ...Delete: {f}")
            except Exception as e:
                print(f"  DeleteFailed {f}: {e}")
        
        try:
            os.rmdir(temp_dir)
            print(f"  ...Delete...: {temp_dir}")
        except Exception as e:
            print(f"  Delete...Failed: {e}")
        
    else:
        print("❌ ...Generate...")# generate_train_data_2022_2025.py


In [ ]:
import os
import re
import time
import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pickle
import gzip
import gc
from datetime import datetime, date
import warnings
import torch.nn as nn  
import json
warnings.filterwarnings('ignore')

# ---------------- Config ----------------
BASE_DIR = r"."
RESULT_FILE = os.path.join(BASE_DIR, "2019_train", "t0_c2_result", "All_AUC_Report_Year_Train_2019.txt")
MODEL_DIR = os.path.join(BASE_DIR, "2019_train", "t0_c2_net")
OUTPUT_DIR = os.path.join(BASE_DIR, "data_for_train_2025")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Code comment
FEATURE_FOLDER = os.path.join(BASE_DIR, "data_for_features")
DYNAMIC_GRAPH_FILE = os.path.join(BASE_DIR, "data_concept_graph", "full_dynamic_graph.parquet")
CONCEPT_LIST_FILE = r"./data/entities.txt"  # Code comment

# Code comment
FEATURE_FILE = os.path.join(BASE_DIR, "data_eval", "eval_data_pair_feature.parquet")
ID_FILE = os.path.join(BASE_DIR, "data_eval", "data_eval_pair_solution.parquet")

# Code comment
YEAR_START = 2022  # Code comment
YEAR_END = 2025    # Code comment
YEARS_DELTA = 3    # Code comment

# Code comment
BATCH_SIZE = 1000000  # Code comment
MAX_PAIRS = None    # Code comment

# Code comment
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "checkpoint.json")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.txt")

  # Comment
class ff_network(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ff_network, self).__init__()
        self.semnet = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        return self.semnet(x)


  # Comment
def log_progress(message, print_to_console=True):
    """...Progress..."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_message = f"[{timestamp}] {message}"
    
    if print_to_console:
        print(log_message)
    
    with open(PROGRESS_FILE, 'a', encoding='utf-8') as f:
        f.write(log_message + "\n")


def save_checkpoint(batch_num, processed_pairs, batch_files, total_found_in_map, total_not_found):
    """SaveCheck..."""
    checkpoint = {
        'batch_num': batch_num,
        'processed_pairs': processed_pairs,
        'batch_files': batch_files,
        'total_found_in_map': total_found_in_map,
        'total_not_found': total_not_found,
        'timestamp': datetime.now().isoformat()
    }
    
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, indent=2)
    
    log_progress(f"💾 Check...Save: ... {batch_num}, Processed {processed_pairs}  items...")


def load_checkpoint():
    """LoadCheck..."""
    if not os.path.exists(CHECKPOINT_FILE):
        return None
    
    try:
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        log_progress(f"🔍 FoundCheck...: ... {checkpoint['batch_num']}, Processed {checkpoint['processed_pairs']}  items...")
        return checkpoint
    except Exception as e:
        log_progress(f"⚠️ LoadCheck...Failed: {e}")
        return None


def find_best_ir(result_file_path):
    """...Result...Found...IR...eval AUC..."""
    pattern = re.compile(r'IR=\[(\d+)\]: train=([\d.]+); test=([\d.]+); eval=([\d.]+)')
    results = []
    
    log_progress(f"🔎 ...Result...: {result_file_path}")
    with open(result_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                ir, train, test, eval_ = m.groups()
                results.append({
                    'IR': int(ir),
                    'train': float(train),
                    'test': float(test),
                    'eval': float(eval_)
                })
    
    if not results:
        raise RuntimeError(f"... {result_file_path} ...IR...Check...")
    
    df = pd.DataFrame(results)
    best_idx = df['eval'].idxmax()
    best_ir = int(df.loc[best_idx, 'IR'])
    best_eval = float(df.loc[best_idx, 'eval'])
    
    log_progress(f"✅ Found...IR={best_ir} (eval AUC={best_eval:.4f})")
    log_progress("\n...IR...AUC...:")
    log_progress(df.to_string(index=False))
    
    return best_ir, best_eval, df


def load_feature_mapping(feature_file, id_file):
    """
    Load...ID...Create...
    
    ...:
        feature_map: ...(v1, v2)...141...
    """
    log_progress(f"🔎 Load......")
    
      # Comment
    if not os.path.exists(feature_file):
        raise FileNotFoundError(f"...does not exist: {feature_file}")
    
    log_progress(f"Load...: {feature_file}")
    df_features = pd.read_parquet(feature_file)
    log_progress(f"...: {df_features.shape}")
    
      # Comment
    if not os.path.exists(id_file):
        raise FileNotFoundError(f"ID...does not exist: {id_file}")
    
    log_progress(f"LoadID...: {id_file}")
    df_ids = pd.read_parquet(id_file)
    
    # Code comment
    if 'v1' in df_ids.columns and 'v2' in df_ids.columns:
        df_ids = df_ids[['v1', 'v2']]
    else:
        # Code comment
        df_ids = df_ids.iloc[:, :2]
        df_ids.columns = ['v1', 'v2']
    
    log_progress(f"ID...: {df_ids.shape}")
    
      # Comment
    if len(df_features) != len(df_ids):
        raise ValueError(f"...({len(df_features)} rows)...ID...({len(df_ids)} rows) rows...")
    
      # Comment
    df_combined = pd.concat([df_ids.reset_index(drop=True), 
                             df_features.reset_index(drop=True)], axis=1)
    
      # Comment
    feature_map = {}
    
    log_progress("Create......")
    for idx, row in df_combined.iterrows():
        v1, v2 = int(row['v1']), int(row['v2'])
        # Code comment
        if v1 > v2:
            v1, v2 = v2, v1
        
          # Comment
        features = row.iloc[2:].values.astype(np.float32)
        
        # Code comment
        if len(features) != 141:
            log_progress(f"⚠️ Warning: ... {len(features)}...141")
        
        feature_map[(v1, v2)] = features
        
        if (idx + 1) % 1000000 == 0:
            log_progress(f"  Processed {idx+1}  items...")
    
    log_progress(f"✅ SuccessfullyCreate {len(feature_map)}  items...")
    
      # Comment
    if len(feature_map) > 0:
        sample_features = next(iter(feature_map.values()))
        log_progress(f"...: {len(sample_features)}")
        log_progress(f"...: [{sample_features.min():.4f}, {sample_features.max():.4f}]")
        log_progress(f"...: {sample_features.mean():.4f}")
    
    return feature_map


def get_unconnected_pairs_batch(graph_file, year_start, batch_size=100000, max_pairs=None, start_from=0):
    """
    ...year_start...
    
    ...: Generate... (batch_pairs, total_count, current_index)
    """
    log_progress(f"🔎 ... {year_start}......")
    
      # Comment
    pf = pq.ParquetFile(graph_file)
    
    # Code comment
    existing_pairs = set()
    total_edges = 0
    
    log_progress("...already exists......")
    for batch in pf.iter_batches(batch_size=batch_size):
        df_batch = batch.to_pandas()
        
          # Comment
        time_col = None
        for col in df_batch.columns:
            if 'time' in col.lower() or 'date' in col.lower() or 'year' in col.lower():
                time_col = col
                break
        
        # Code comment
        if time_col is not None:
            try:
                # Code comment
                if pd.api.types.is_integer_dtype(df_batch[time_col]):
                    df_batch = df_batch[df_batch[time_col] <= year_start]
                # Code comment
                elif pd.api.types.is_string_dtype(df_batch[time_col]):
                    year_start_str = str(year_start)
                    df_batch = df_batch[df_batch[time_col].str.startswith(year_start_str) | 
                                         (df_batch[time_col] < str(year_start + 1))]
                # Code comment
                else:
                    year_start_date = pd.Timestamp(f"{year_start}-12-31")
                    df_batch = df_batch[pd.to_datetime(df_batch[time_col]) <= year_start_date]
            except Exception as e:
                log_progress(f"  ...: {e}...Skip...")
        
          # Comment
        for _, row in df_batch.iterrows():
            v1, v2 = row['v1'], row['v2']
            # Code comment
            if v1 < v2:
                existing_pairs.add((v1, v2))
            else:
                existing_pairs.add((v2, v1))
        
        total_edges += len(df_batch)
        if len(existing_pairs) % 1000000 == 0:
            log_progress(f"  ... {len(existing_pairs)}  items...")
    
    log_progress(f"✅ ... {len(existing_pairs)}  itemsalready exists...{year_start}...")
    
      # Comment
    # Code comment
    concepts = set()
    for year in [year_start, year_start-1, year_start-2]:
        cite_file = os.path.join(FEATURE_FOLDER, f"concept_node_citation_data_{year}.parquet")
        if os.path.exists(cite_file):
            try:
                df_cite = pd.read_parquet(cite_file)
                # Code comment
                concepts.update(df_cite.iloc[:, 0].tolist())
            except Exception as e:
                log_progress(f"  Load {year}...Failed: {e}")
    
    concepts = list(concepts)
    if len(concepts) == 0:
        # Code comment
        for v1, v2 in existing_pairs:
            concepts.add(v1)
            concepts.add(v2)
        concepts = list(concepts)
    
    log_progress(f"✅ ... {len(concepts)}  items...")
    
      # Comment
    from itertools import combinations
    
    total_possible = len(concepts) * (len(concepts) - 1) // 2
    log_progress(f"📊  [description] : {total_possible:,}")
    
    # Code comment
    unconnected_batch = []
    batch_count = 0
    total_unconnected = 0
    skipped = 0
    
    for i, (v1, v2) in enumerate(combinations(concepts, 2)):
        # Code comment
        if i < start_from:
            continue
        
        # Code comment
        if v1 > v2:
            v1, v2 = v2, v1
        
          # Comment
        if (v1, v2) not in existing_pairs:
            unconnected_batch.append((v1, v2))
            total_unconnected += 1
        
        # Code comment
        if len(unconnected_batch) >= batch_size or (max_pairs and total_unconnected >= max_pairs):
            if unconnected_batch:  # Code comment
                batch_count += 1
                log_progress(f"  Generate... {batch_count}: {len(unconnected_batch)}  items... (... {i+1} Start)")
                yield unconnected_batch, total_unconnected, i + 1
                unconnected_batch = []
        
        # Code comment
        if max_pairs and total_unconnected >= max_pairs:
            break
        
        # Code comment
        if (i + 1) % 1000000 == 0:
            log_progress(f"  Processed {i+1:,}  items...Found {total_unconnected:,}  items...")
    
    # Code comment
    if unconnected_batch:
        batch_count += 1
        log_progress(f"  Generate... {batch_count}: {len(unconnected_batch)}  items...")
        yield unconnected_batch, total_unconnected, i + 1
    
    log_progress(f"✅ ...Found {total_unconnected:,}  items...")


def get_label_for_pair(v1, v2, graph_file, year_end, IR):
    """
    ...year_end...
    
    ...year_end... ≥ IR...=1...=0
    """
    try:
        pf = pq.ParquetFile(graph_file)
    except Exception as e:
        log_progress(f"  ...Open...: {e}", print_to_console=False)
        return 0, 0
    
    total_citations = 0
    
    # Code comment
    for batch in pf.iter_batches(batch_size=100000):
        try:
            df_batch = batch.to_pandas()
        except:
            continue
        
        # Code comment
        mask = ((df_batch['v1'] == v1) & (df_batch['v2'] == v2)) | \
               ((df_batch['v1'] == v2) & (df_batch['v2'] == v1))
        
        if mask.any():
            df_pair = df_batch[mask]
            
            # Code comment
            cite_col = None
            for col in df_pair.columns:
                if col.lower().startswith('c') and col[1:].isdigit():
                    year = int(col[1:])
                    if year <= year_end:
                        cite_col = col
                        break
                elif col.lower() in ['ct', 'citation', 'citations']:
                    cite_col = col
            
            if cite_col is not None:
                total_citations += df_pair[cite_col].sum()
    
    label = 1 if total_citations >= IR else 0
    return label, total_citations


def extract_features_from_mapping(v1, v2, feature_map):
    """
    ...Extract...(v1, v2)...141...
    
    Parameters:
        v1, v2: ...ID
        feature_map: ...(v1, v2)...141...
    
    ...:
        141... columns [description] None
    """
    # Code comment
    if v1 > v2:
        v1, v2 = v2, v1
    
    # Code comment
    if (v1, v2) in feature_map:
        features = feature_map[(v1, v2)]
        return features.tolist() if isinstance(features, np.ndarray) else features
    else:
        return None


# ---------------- Main ----------------
def main():
    start_time_total = time.time()
    
      # Comment
    checkpoint = load_checkpoint()
    
      # Comment
    log_progress("=" * 60)
    log_progress("...1: ...IR...")
    log_progress("=" * 60)
    best_ir, best_eval, df_results = find_best_ir(RESULT_FILE)
    
      # Comment
    log_progress("\n" + "=" * 60)
    log_progress("...2: Load...")
    log_progress("=" * 60)
    feature_map = load_feature_mapping(FEATURE_FILE, ID_FILE)
    
      # Comment
    output_file = os.path.join(OUTPUT_DIR, f"train_data_{YEAR_START}_{YEAR_END}_IR{best_ir}.parquet")
    temp_dir = os.path.join(OUTPUT_DIR, "temp")
    os.makedirs(temp_dir, exist_ok=True)
    
    log_progress("\n" + "=" * 60)
    log_progress(f"...3: Generate...")
    log_progress(f"Output...: {output_file}")
    log_progress(f"...: {YEAR_START}")
    log_progress(f"...: {YEAR_END}")
    log_progress(f"IR...: {best_ir}")
    log_progress("=" * 60)
    
      # Comment
    log_progress("\n📊 Start...Process......")
    
    # Code comment
    if checkpoint:
        start_batch = checkpoint['batch_num']
        batch_files = checkpoint['batch_files']
        total_processed = checkpoint['processed_pairs']
        total_found_in_map = checkpoint['total_found_in_map']
        total_not_found = checkpoint['total_not_found']
        log_progress(f"🔄 ...Check...: ... {start_batch} ResumeProcess")
        log_progress(f"   ...Temp file: {len(batch_files)}  items")
        log_progress(f"   Processed...: {total_processed}")
        log_progress(f"   ...Found...: {total_found_in_map}")
        log_progress(f"   ...Found...: {total_not_found}")
    else:
        start_batch = 0
        batch_files = []
        total_processed = 0
        total_found_in_map = 0
        total_not_found = 0
    
    # Code comment
    pair_generator = get_unconnected_pairs_batch(
        DYNAMIC_GRAPH_FILE, 
        YEAR_START, 
        batch_size=BATCH_SIZE,
        max_pairs=MAX_PAIRS
    )
    
    current_batch = 0
    for batch_pairs, total_found, current_index in pair_generator:
        current_batch += 1
        
        # Code comment
        if current_batch <= start_batch:
            log_progress(f"⏭️ SkipProcessed... {current_batch}")
            continue
        
        batch_start = time.time()
        
        log_progress(f"\n--- Process... {current_batch} ---")
        log_progress(f"...: {len(batch_pairs)}")
        log_progress(f"CumulativeFound...: {total_found}")
        log_progress(f"...: {current_index}")
        
          # Comment
        batch_data = []
        batch_found = 0
        batch_not_found = 0
        
        for i, (v1, v2) in enumerate(batch_pairs):
            if (i + 1) % 1000 == 0:  # Code comment
                log_progress(f"  Process... {i+1}/{len(batch_pairs)}  items...")
            
            # Code comment
            features = extract_features_from_mapping(v1, v2, feature_map)
            
            if features is not None:
                # Code comment
                label, citations = get_label_for_pair(v1, v2, DYNAMIC_GRAPH_FILE, YEAR_END, best_ir)
                
                  # Comment
                row = [v1, v2, label] + features
                batch_data.append(row)
                
                batch_found += 1
                total_found_in_map += 1
            else:
                batch_not_found += 1
                total_not_found += 1
            
            total_processed += 1
        
          # Comment
        if batch_data:
            df_batch = pd.DataFrame(batch_data)
              # Comment
            col_names = ['v1', 'v2', 'label'] + [f'f{i}' for i in range(141)]
            df_batch.columns = col_names
            
            temp_file = os.path.join(temp_dir, f"batch_{current_batch:04d}.parquet")
            df_batch.to_parquet(temp_file, index=False)
            batch_files.append(temp_file)
            
            batch_time = time.time() - batch_start
            log_progress(f"✅ ... {current_batch} Done... {batch_time:.2f}s...Save... {temp_file}")
            log_progress(f"   ...Found: {batch_found}  items...Found: {batch_not_found}  items")
            log_progress(f"   ...: {df_batch['label'].value_counts().to_dict()}")
            
              # Comment
            save_checkpoint(current_batch, total_processed, batch_files, total_found_in_map, total_not_found)
            
            # Code comment
            del df_batch, batch_data
            gc.collect()
        else:
            log_progress(f"⚠️ ... {current_batch} ...Found...")
    
    log_progress(f"\n📊 Statistics...:")
    log_progress(f"   ...Process...: {total_processed}")
    log_progress(f"   ...Found: {total_found_in_map}")
    log_progress(f"   ...Found: {total_not_found}")
    if total_processed > 0:
        log_progress(f"   Found...: {total_found_in_map/total_processed*100:.2f}%")
    
      # Comment
    if batch_files:
        log_progress("\n" + "=" * 60)
        log_progress("...4: Merge...")
        log_progress("=" * 60)
        
          # Comment
        dfs = []
        for i, f in enumerate(batch_files):
            log_progress(f"Read... {i+1}/{len(batch_files)}: {f}")
            df = pd.read_parquet(f)
            dfs.append(df)
        
        log_progress(f"Merge {len(dfs)}  itemsDataFrame...")
        df_final = pd.concat(dfs, ignore_index=True)
        
        # Code comment
        df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
        
        log_progress(f"...Size: {len(df_final)}  rows, {len(df_final.columns)}  columns")
        log_progress(f"...:")
        log_progress(str(df_final['label'].value_counts()))
        log_progress(f"...: {df_final['label'].mean():.4f}")
        
          # Comment
        log_progress(f"Save...: {output_file}")
        df_final.to_parquet(output_file, index=False)
        
          # Comment
        sample_file = output_file.replace('.parquet', '_sample.csv')
        df_final.head(1000).to_csv(sample_file, index=False)
        log_progress(f"...Save...: {sample_file}")
        
        # CleanTemp file
        log_progress("CleanTemp file...")
        for f in batch_files:
            try:
                os.remove(f)
                log_progress(f"  ...Delete: {f}")
            except Exception as e:
                log_progress(f"  DeleteFailed {f}: {e}")
        
        try:
            os.rmdir(temp_dir)
            log_progress(f"  ...Delete...: {temp_dir}")
        except Exception as e:
            log_progress(f"  Delete...Failed: {e}")
        
          # Comment
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
            log_progress("✅ ...DeleteCheck...")
        
    else:
        log_progress("❌ ...Generate...")
    
    total_time = time.time() - start_time_total
    log_progress("\n" + "=" * 60)
    log_progress(f"✅ ...Done...Total time: {total_time:.2f}s")
    log_progress(f"Output...: {output_file}")
    log_progress("=" * 60)


if __name__ == "__main__":
    main()
    
    total_time = time.time() - start_time_total
    print("\n" + "=" * 60)
    print(f"✅ ...Done...Total time: {total_time:.2f}s")
    print(f"Output...: {output_file}")
    print("=" * 60)


if __name__ == "__main__":
    main()

In [ ]:
import os
import re
import time
import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pickle
import gzip
import gc
from datetime import datetime, date
import warnings
import torch.nn as nn  
import json
warnings.filterwarnings('ignore')

# ---------------- Config ----------------
BASE_DIR = r"."
RESULT_FILE = os.path.join(BASE_DIR, "2016_train", "t0_c2_result", "All_AUC_Report_Year_Train_2016.txt")
MODEL_DIR = os.path.join(BASE_DIR, "2016_train", "t0_c2_net")
OUTPUT_DIR = os.path.join(BASE_DIR, "data_for_train")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Code comment
FEATURE_FOLDER = os.path.join(BASE_DIR, "data_for_features")
DYNAMIC_GRAPH_FILE = os.path.join(BASE_DIR, "data_concept_graph", "full_dynamic_graph.parquet")
CONCEPT_LIST_FILE = r"./data/entities.txt" # Code comment

# Code comment
FEATURE_FILE = os.path.join(BASE_DIR, "data_eval", "eval_data_pair_feature.parquet")
ID_FILE = os.path.join(BASE_DIR, "data_eval", "data_eval_pair_solution.parquet")

# Code comment
YEAR_START = 2016  # Code comment
YEAR_END = 2019    # Code comment
YEARS_DELTA = 3    # Code comment

# Code comment
BATCH_SIZE = 1000000  # Code comment
MAX_PAIRS = None    # Code comment

# Code comment
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "checkpoint.json")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.txt")

  # Comment
class ff_network(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ff_network, self).__init__()
        self.semnet = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        return self.semnet(x)


  # Comment
def log_progress(message, print_to_console=True):
    """...Progress..."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_message = f"[{timestamp}] {message}"
    
    if print_to_console:
        print(log_message)
    
    with open(PROGRESS_FILE, 'a', encoding='utf-8') as f:
        f.write(log_message + "\n")


def save_checkpoint(batch_num, processed_pairs, batch_files, total_found_in_map, total_not_found):
    """SaveCheck..."""
    checkpoint = {
        'batch_num': batch_num,
        'processed_pairs': processed_pairs,
        'batch_files': batch_files,
        'total_found_in_map': total_found_in_map,
        'total_not_found': total_not_found,
        'timestamp': datetime.now().isoformat()
    }
    
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, indent=2)
    
    log_progress(f"💾 Check...Save: ... {batch_num}, Processed {processed_pairs}  items...")


def load_checkpoint():
    """LoadCheck..."""
    if not os.path.exists(CHECKPOINT_FILE):
        return None
    
    try:
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        log_progress(f"🔍 FoundCheck...: ... {checkpoint['batch_num']}, Processed {checkpoint['processed_pairs']}  items...")
        return checkpoint
    except Exception as e:
        log_progress(f"⚠️ LoadCheck...Failed: {e}")
        return None


def find_best_ir(result_file_path):
    """...Result...Found...IR...eval AUC..."""
    pattern = re.compile(r'IR=\[(\d+)\]: train=([\d.]+); test=([\d.]+); eval=([\d.]+)')
    results = []
    
    log_progress(f"🔎 ...Result...: {result_file_path}")
    with open(result_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                ir, train, test, eval_ = m.groups()
                results.append({
                    'IR': int(ir),
                    'train': float(train),
                    'test': float(test),
                    'eval': float(eval_)
                })
    
    if not results:
        raise RuntimeError(f"... {result_file_path} ...IR...Check...")
    
    df = pd.DataFrame(results)
    best_idx = df['eval'].idxmax()
    best_ir = int(df.loc[best_idx, 'IR'])
    best_eval = float(df.loc[best_idx, 'eval'])
    
    log_progress(f"✅ Found...IR={best_ir} (eval AUC={best_eval:.4f})")
    log_progress("\n...IR...AUC...:")
    log_progress(df.to_string(index=False))
    
    return best_ir, best_eval, df


def load_feature_mapping(feature_file, id_file):
    """
    Load...ID...Create...
    
    ...:
        feature_map: ...(v1, v2)...141...
    """
    log_progress(f"🔎 Load......")

      # Comment
    if not os.path.exists(feature_file):
        raise FileNotFoundError(f"...does not exist: {feature_file}")
    
    log_progress(f"Load...: {feature_file}")
    df_features = pd.read_parquet(feature_file)
    log_progress(f"...: {df_features.shape}")
    
      # Comment
    if not os.path.exists(id_file):
        raise FileNotFoundError(f"ID...does not exist: {id_file}")
    
    log_progress(f"LoadID...: {id_file}")
    df_ids = pd.read_parquet(id_file)
    
    # Code comment
    if 'v1' in df_ids.columns and 'v2' in df_ids.columns:
        df_ids = df_ids[['v1', 'v2']]
    else:
        # Code comment
        df_ids = df_ids.iloc[:, :2]
        df_ids.columns = ['v1', 'v2']
    
    log_progress(f"ID...: {df_ids.shape}")
    
      # Comment
    if len(df_features) != len(df_ids):
        raise ValueError(f"...({len(df_features)} rows)...ID...({len(df_ids)} rows) rows...")
    
      # Comment
    df_combined = pd.concat([df_ids.reset_index(drop=True), 
                             df_features.reset_index(drop=True)], axis=1)
    
      # Comment
    feature_map = {}
    
    log_progress("Create......")
    for idx, row in df_combined.iterrows():
        v1, v2 = int(row['v1']), int(row['v2'])
        # Code comment
        if v1 > v2:
            v1, v2 = v2, v1
        
          # Comment
        features = row.iloc[2:].values.astype(np.float32)
        
        # Code comment
        if len(features) != 141:
            log_progress(f"⚠️ Warning: ... {len(features)}...141")
        
        feature_map[(v1, v2)] = features
        
        if (idx + 1) % 1000000 == 0:
            log_progress(f"  Processed {idx+1}  items...")
    
    log_progress(f"✅ SuccessfullyCreate {len(feature_map)}  items...")
    
      # Comment
    if len(feature_map) > 0:
        sample_features = next(iter(feature_map.values()))
        log_progress(f"...: {len(sample_features)}")
        log_progress(f"...: [{sample_features.min():.4f}, {sample_features.max():.4f}]")
        log_progress(f"...: {sample_features.mean():.4f}")
    
    return feature_map


def get_labels_from_solution_files(unconnected_connected_file, unconnected_unconnected_file, feature_map):
    """
     [description] Generate...2019...1...0...
    """
    # Code comment
    connected_df = pd.read_parquet(unconnected_connected_file)
    connected_pairs = set(zip(connected_df['v1'], connected_df['v2']))

    # Code comment
    unconnected_df = pd.read_parquet(unconnected_unconnected_file)
    unconnected_pairs = set(zip(unconnected_df['v1'], unconnected_df['v2']))

    # Code comment
    labels = []
    for v1, v2 in connected_pairs:
        features = extract_features_from_mapping(v1, v2, feature_map)
        if features is not None:
            labels.append([v1, v2, 1] + features.tolist())

    for v1, v2 in unconnected_pairs:
        features = extract_features_from_mapping(v1, v2, feature_map)
        if features is not None:
            labels.append([v1, v2, 0] + features.tolist())

    return labels

# Code comment
def main():
    start_time_total = time.time()
    
      # Comment
    checkpoint = load_checkpoint()
    
      # Comment
    log_progress("=" * 60)
    log_progress("...1: ...IR...")
    log_progress("=" * 60)
    best_ir, best_eval, df_results = find_best_ir(RESULT_FILE)
    
      # Comment
    log_progress("\n" + "=" * 60)
    log_progress("...2: Load...")
    log_progress("=" * 60)
    feature_map = load_feature_mapping(FEATURE_FILE, ID_FILE)
    
      # Comment
    log_progress("\n" + "=" * 60)
    log_progress("...3: Generate...")
    log_progress("=" * 60)
    unconnected_connected_file = os.path.join(BASE_DIR, "data_pair_solution", "unconnected_2016_pair_solution_connected_2019.parquet")
    unconnected_unconnected_file = os.path.join(BASE_DIR, "data_pair_solution", "unconnected_2016_pair_solution_unconnected_2019.parquet")
    
    labels_data = get_labels_from_solution_files(unconnected_connected_file, unconnected_unconnected_file, feature_map)
    
      # Comment
    output_file = os.path.join(OUTPUT_DIR, f"train_data_{YEAR_START}_{YEAR_END}_IR{best_ir}.parquet")
    log_progress(f"Output...: {output_file}")
    
    df_labels = pd.DataFrame(labels_data)
    col_names = ['v1', 'v2', 'label'] + [f'f{i}' for i in range(141)]
    df_labels.columns = col_names
    
    df_labels.to_parquet(output_file, index=False)
    
    log_progress(f"✅ Generate...Successfully...Save... {output_file}")
    
    total_time = time.time() - start_time_total
    log_progress("\n" + "=" * 60)
    log_progress(f"✅ ...Done...Total time: {total_time:.2f}s")
    log_progress("=" * 60)


if __name__ == "__main__":
    main()
    
    total_time = time.time() - start_time_total
    print("\n" + "=" * 60)
    print(f"✅ ...Done...Total time: {total_time:.2f}s")
    print("=" * 60)